# Character-Level Text Modeling - Dataset Exploration

This notebook is meant for lecture use. It keeps things simple:
- load a character-level text dataset
- inspect the raw text
- build the character vocabulary
- create integer encodings
- visualize sequence patterns

Recommended dataset: **Tiny Shakespeare**

How to read the code in this notebook:
- each code cell performs one small preprocessing step
- we move from raw text to numbers because neural networks cannot work directly on characters
- after each step, pay attention to both the variable names and the printed output, because together they show how text is converted into training data

In the first code cell, we import plotting tools and shared helper functions from our local package. We also make sure Python can find the project root so the notebook can import those helpers whether Jupyter was started from the repository root or from inside `src/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "nirma_university_language_models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt

from nirma_university_language_models import (
    build_vocabulary,
    decode_ids,
    encode_text,
    load_tinyshakespeare_text,
    make_input_target_pair,
    make_sequences,
    most_common_characters,
)


## 1. Download the dataset

We will use Tiny Shakespeare because it is:
- small
- clean
- easy to train on
- perfect for showing RNN vs GRU vs LSTM behavior

What the next code cells do:
- `load_tinyshakespeare_text()` returns the text and the file path being used
- this keeps the notebook simple because the download and file-handling logic already lives in our shared helper module
- after loading, we print the first 1000 characters so students can see the raw corpus before any preprocessing

This is an important habit in machine learning: inspect the raw data first before building a model.

In [ ]:
text, DATA_PATH = load_tinyshakespeare_text()

print("Saved to:", DATA_PATH)

In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))
print("\nFirst 1000 characters:\n")
print(text[:1000])

## 2. Character vocabulary

Here we build the set of unique characters in the corpus.

Why this matters:
- the model does not understand letters directly
- every character must be assigned an integer ID
- `char_to_idx` maps a character like `'a'` to a number
- `idx_to_char` does the reverse mapping so we can convert predictions back into readable text

When students run the next cell, they should notice that even spaces, punctuation, and newline characters are part of the vocabulary.

In [ ]:
chars, char_to_idx, idx_to_char = build_vocabulary(text)
vocab_size = len(chars)

print("Vocabulary size:", vocab_size)
print(chars)

## 3. Character frequency

Now we measure how often each character appears.

What the code is doing:
- `most_common_characters(...)` counts the characters in the full corpus
- the first code cell shows the raw counts as Python data
- the next code cell turns those counts into a bar chart

This helps students connect the dataset to the model. For example, if spaces and common letters dominate the corpus, the model will see those symbols much more often during training.

In [ ]:
most_common = most_common_characters(text, limit=20)
most_common

In [ ]:
labels = [repr(ch) for ch, _ in most_common]
values = [count for _, count in most_common]

plt.figure(figsize=(12, 4))
plt.bar(labels, values)
plt.title("Top 20 Most Frequent Characters")
plt.xlabel("Character")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Encode the text as integers

This is the key transition from text preprocessing to model-ready data.

The next code cell uses `encode_text(...)` to replace every character with its vocabulary index. The output is a NumPy array, which is much easier to slice, batch, and later convert into PyTorch tensors.

Students should notice:
- the shape tells us how many total characters are in the corpus
- the printed values are no longer readable text, but they preserve the same sequence information

In [ ]:
encoded = encode_text(text, char_to_idx)

print("Encoded shape:", encoded.shape)
print("First 50 encoded values:")
print(encoded[:50])

## 5. Where does embedding dimension come from?

Students often mix up `vocab_size` and `embedding_dim`, but they mean different things.

- `vocab_size` comes from the dataset because it is the number of unique characters
- `embedding_dim` does not come from the dataset automatically
- we choose `embedding_dim` as a model design choice such as 8, 16, 32, or 64
- the embedding layer therefore has shape `(vocab_size, embedding_dim)`

So if this corpus has `vocab_size = 65` and we choose `embedding_dim = 16`, then the model learns a table with 65 rows and 16 columns. Each character ID selects one 16-dimensional vector from that table.

In [ ]:
import torch

EMBED_DIM = 16
embedding = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=EMBED_DIM)
sample_char_ids = torch.tensor(encoded[:8], dtype=torch.long)
sample_embeddings = embedding(sample_char_ids)

print("Embedding table shape:", tuple(embedding.weight.shape))
print("Input ID shape:", tuple(sample_char_ids.shape))
print("Output embedding shape:", tuple(sample_embeddings.shape))
print("First character embedding vector:", sample_embeddings[0])

## 6. Simple decoding check

A good preprocessing pipeline should be reversible.

In the next code cell, we take the first few encoded integers and convert them back into characters with `decode_ids(...)`. If the decoded text matches the original text, then our vocabulary and encoding steps are working correctly.

In [ ]:
sample_ids = encoded[:200]
decoded = decode_ids(sample_ids, idx_to_char)
print(decoded)

## 7. Build input-target pairs

For next-character prediction:
- input = a chunk of characters
- target = the same chunk shifted by one character

The next code cell creates one training example.

What students should observe:
- `x` is the input sequence the model reads
- `y` is the correct answer sequence
- every target character is the next character after the corresponding input character

This is the core learning task for the language model: given the current character history, predict what comes next.

In [ ]:
seq_len = 80
x, y = make_input_target_pair(encoded, seq_len=seq_len)

print("Input text:")
print(decode_ids(x, idx_to_char))
print("\nTarget text:")
print(decode_ids(y, idx_to_char))

## 8. Visualize a few training sequences

One example is useful, but training needs many examples.

The helper function in the next code cell slices the encoded corpus into several fixed-length sequences. We print both `X` and `Y` for a few examples so students can see that the shift-by-one rule is consistent across the dataset.

This section connects the preprocessing pipeline to batching later in the training notebook.

In [ ]:
xs, ys = make_sequences(encoded, seq_len=60, step=60, n_examples=4)

for i, (x_seq, y_seq) in enumerate(zip(xs, ys), start=1):
    print(f"Example {i}")
    print("X:", decode_ids(x_seq, idx_to_char))
    print("Y:", decode_ids(y_seq, idx_to_char))
    print("-" * 80)